# Data Features Selection
This notebook is the base for features  selection
## Used libraries

In [1]:
# type: ignore
import seaborn as sns
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import RFE, SelectKBest, SelectFromModel, f_classif, mutual_info_classif
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA 
from sklearn.preprocessing import StandardScaler 
from sklearn.model_selection import cross_val_score 

from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from statsmodels.stats.multitest import multipletests

## Loading data
- Cleaning infinite values and NaNs after the previous feature augmentation
- Looking for potential highly correlated features

In [2]:
train = pd.read_csv('../train_extended.csv', index_col='ID')
test = pd.read_csv('../test_extended.csv', index_col='ID')

## Feature selection
Several approch were used here :
- Trees Feature Importance
- RFE
- Statistical test
- Dimension reduction
- SFS

In [8]:
x_train, y_train = train.drop('RET', axis=1), train['RET']

### Tree Feature Importance method

In [ ]:
# Feature selection through prefit model and SelectFromModel
model = RandomForestClassifier(n_estimators=500, max_depth=8,  n_jobs=-1, verbose=1)
model.fit(x_train, y_train)

# feature importance of the model
importances = model.feature_importances_

# plot the feature importances sorted
indices = np.argsort(importances)[::-1]
sns.barplot(x=importances[indices], y=x_train.columns[indices], orient='h')

In [ ]:
# save the features importance of the model ranked by importance with their names
features_importance = pd.DataFrame({'feature': x_train.columns, 'importance': model.feature_importances_}).sort_values(by='importance', ascending=False)
features_importance.to_csv('features_importance.csv')

In [ ]:
# Select 40 features with SelectFromModel with the prefit model
pre_selector = SelectFromModel(model, max_features=50, prefit=True)
pre_selectedFeatures = x_train.columns[pre_selector.get_support()]

pre_selectedFeatures

### Recursive Feature Elimination

In [ ]:
# Procedding to RFE forward selection from the selected features
# We will use the RandomForestClassifier as estimator
estimator = XGBClassifier(n_estimators=500, max_depth=8, n_jobs=-1)
selector = RFE(estimator, n_features_to_select=45, step=1, verbose=10)
selector.fit(x_train[pre_selectedFeatures], y_train)
selectedFeaturesXGB = pre_selectedFeatures[selector.support_]  # Get the selected features

In [ ]:
estimator2 = RandomForestClassifier(n_estimators=500, max_depth=8, n_jobs=-1)
selector2 = RFE(estimator2, n_features_to_select=45, step=1, verbose=10)
selector2.fit(x_train[pre_selectedFeatures], y_train)
selectedFeaturesRF = pre_selectedFeatures[selector2.support_]  # Get the selected features

In [ ]:
# save the selected features of the RFE XGB and RF by transforming them to a DataFrame then save them to a csv file
selectedFeaturesXGB = pd.DataFrame(selectedFeaturesXGB, columns=['feature'])
selectedFeaturesRF = pd.DataFrame(selectedFeaturesRF, columns=['feature'])
selectedFeaturesXGB.to_csv('selectedFeaturesXGB.csv')
selectedFeaturesRF.to_csv('selectedFeaturesRF.csv')

### Statistical significativity tests

In [ ]:
# Selection of features with SelectKBest f_classif
pipe_selection = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('selector', SelectKBest(score_func=f_classif, k='all'))
])

pipe_selection.fit(x_train, y_train)
scores = pipe_selection.named_steps['selector'].scores_

# put the scores in a DataFrame
scores_f = pd.DataFrame({'feature': x_train.columns, 'score': scores})
scores_f = scores_f.sort_values(by='score', ascending=False)
scores_f.to_csv('../scores_f.csv')

In [ ]:
# Selection of features with SelectKBest mutual_info_classif
pipe_selection = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('selector', SelectKBest(score_func=mutual_info_classif, k='all'))
])

pipe_selection.fit(x_train, y_train)
scores_mutual_info = pipe_selection.named_steps['selector'].scores_

# put the scores in a DataFrame
scores_mutual_info_f = pd.DataFrame({'feature': x_train.columns, 'score': scores_mutual_info})
scores_mutual_info_f = scores_mutual_info_f.sort_values(by='score', ascending=False)
scores_mutual_info_f.to_csv('../scores_mutual_info.csv')

In [ ]:

def feature_selection_classification(df, target_col, mi_threshold=0.01, pval_threshold=0.05, corr_threshold=0.9):
    """
    Selects the most relevant features for binary classification using:
    1. Mutual Information (MI) to detect nonlinear relationships.
    2. F-Test to compute p-values for statistical significance.
    3. False Discovery Rate (FDR) correction for multiple comparisons.
    4. Correlation filtering to remove redundant features.

    Parameters:
    - df (pd.DataFrame): Dataset containing features and target.
    - target_col (str): Name of the target variable (binary: 0 or 1).
    - mi_threshold (float): Minimum MI score for feature relevance.
    - pval_threshold (float): Maximum corrected p-value for feature selection.
    - corr_threshold (float): Maximum allowed correlation (default 0.9).

    Returns:
    - selected_features (list): List of chosen feature names.
    - df_selected (pd.DataFrame): New dataframe with selected features.
    """

    # Separate features and target
    X = df.drop(columns=[target_col])
    y = df[target_col]

    # Compute Mutual Information for classification
    mi_scores = mutual_info_classif(X, y)
    mi_scores = pd.Series(mi_scores, index=X.columns)
    
    # Compute F-test p-values
    f_scores, p_values = f_classif(X, y)  
    p_values = pd.Series(p_values, index=X.columns)
    
    # Apply False Discovery Rate (FDR) correction for multiple comparisons
    _, p_values_corrected, _, _ = multipletests(p_values, alpha=pval_threshold, method='fdr_bh')
    p_values_corrected = pd.Series(p_values_corrected, index=X.columns)

    # Step 1: Select features based on MI and p-values
    # selected_features = mi_scores[(mi_scores > mi_threshold) & (p_values_corrected < pval_threshold)].index.tolist()

    # Step 2: Remove highly correlated features
    # df_selected = df[selected_features + [target_col]]  # Keep only selected features + target
    # corr_matrix = df_selected.corr().abs()

    # Identify highly correlated pairs
    # upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    # to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > corr_threshold)]

    # Final feature list after removing correlated features
    # selected_features = [f for f in selected_features if f not in to_drop]

    # Return final dataframe with selected features
    # return selected_features, df[selected_features + [target_col]]r
    return mi_scores, p_values, p_values_corrected

mi_scores, p_values, p_values_corrected = feature_selection_classification(train, 'RET')

### Dimension reduction through PCA

In [ ]:
# PCA Pipeline on the whole dataset

pipe_pca = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.99))
])

pipe_pca.fit(x_train)

# draw the explained variance ratio of the PCA
plt.plot(pipe_pca.named_steps['pca'].explained_variance_ratio_)
plt.xlabel('Number of components')
plt.ylabel('Explained variance ratio')
plt.show()


In [ ]:
# train the model with the PCA selected features

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=45)),
    ('model', RandomForestClassifier(n_estimators=500, max_depth=8, n_jobs=-1))
])

scores = cross_val_score(pipe, x_train, y_train, cv=3, scoring='accuracy', verbose=10)
for i, score in enumerate(scores):
    print('Fold %d: %.3f' % (i, score))
print('Accuracy: %.3f +/- %.3f' % (scores.mean(), scores.std()))


### Sequential Feature Slection (forward/backward)

In [40]:

ret_cols = ['RET_%d' % i for i in range(1, 6)]
volume_cols = ['VOLUME_%d' % i for i in range(1, 6)]
SECTOR_cols = [col for col in x_train.columns if 'SECTOR' in col]
WEEK_cols = [col for col in x_train.columns if 'WEEK' in col]

keep_features = list(set(ret_cols + volume_cols + SECTOR_cols + WEEK_cols))  # Features to keep

ban_list = ['DATE', 'SECTOR', 'INDUSTRY', 'STOCK']
candidate_features = [col for col in x_train.columns if col not in keep_features and not any(ban in col for ban in ban_list)] 


In [ ]:
X_fixed = x_train[keep_features]  # Fixed features (features that we want to keep)
X_candidates = x_train[candidate_features]  # Features to test

# Apply the Sequential Feature Selector with a RandomForestClassifier on the candidate features
 
sfs = SequentialFeatureSelector(
    RandomForestClassifier(n_estimators=50, max_depth=6, n_jobs=-1, verbose=3),
    n_features_to_select=1, 
    direction="forward",
    cv=3,
    n_jobs=-1,
)

sfs.fit(X_candidates, y_train)

print("selected features :", list(X_candidates.columns[sfs.get_support()]))

Features sélectionnées : ['Std_RET_20']
